In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
#Load training and test datasets

train = pd.read_csv('/kaggle/input/competitions/home-data-for-ml-course/train.csv')
test = pd.read_csv('/kaggle/input/competitions/home-data-for-ml-course/test.csv')

print ("Train shape"), train.shape
print ("Test shape"), test.shape 

In [ ]:
#plot histogram of sales prices
y = train.SalePrice

y.plot(kind='hist')

In [ ]:
#Create feature set
features = ["GrLivArea", "YearBuilt", "BedroomAbvGr", "FullBath", "LotArea"]
X = train[features]
X.head()

In [ ]:
#create the model
from sklearn.tree import DecisionTreeRegressor

#random_state=1 ensures the same results across runs
housing_model = DecisionTreeRegressor(random_state=1)

#fit model
housing_model.fit(X, y)


In [ ]:
print("Making predictions for the following 5 houses")
print(X.head())

print("The precitions are")
print(housing_model.predict(X.head()))

In [ ]:
from sklearn.metrics import mean_absolute_error

predicted_home_prices = housing_model.predict(X)

mean_absolute_error(y, predicted_home_prices)

In [ ]:
from sklearn.model_selection import train_test_split

# split data into training and validation data, for both features and target 
# The split is based on a random number generator. Supplying a numeric value to
# the random_state argument guarantees we get the same split every time

train_X, val_X, train_y, val_y = train_test_split(X, y, random_state = 0)


In [ ]:

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Build a Random Forest model
forest_model = RandomForestRegressor(
    n_estimators=100,
    random_state=1
)

# Train on the training data
forest_model.fit(train_X, train_y)

# Predict house prices using unseen validation data
forest_predictions = forest_model.predict(val_X)

# Measure how far the predictions are from actual prices
forest_mae = mean_absolute_error(val_y, forest_predictions)

print("Random Forest MAE: $", round(forest_mae, 2))

In [ ]:

from sklearn.model_selection import train_test_split

# Select the house features
features = [
    'GrLivArea',
    'YearBuilt',
    'BedroomAbvGr',
    'FullBath',
    'LotArea'
]

X = train[features]
y = train['SalePrice']

# Split the data into training and validation sets
train_X, val_X, train_y, val_y = train_test_split(
    X, y, random_state=0
)

print("Training data ready!")


In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Train only on the training data
model = DecisionTreeRegressor(random_state=1)
model.fit(train_X, train_y)

# Predict prices for homes held out for validation
predictions = model.predict(val_X)

# Measure performance
mae = mean_absolute_error(val_y, predictions)
r2 = r2_score(val_y, predictions)

print(f"Validation MAE: ${mae:,.0f}")
print(f"Validation R²: {r2:.3f}")

In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest = RandomForestRegressor(
    n_estimators=200,
    random_state=1,
    n_jobs=-1
)
forest.fit(train_X, train_y)

forest_predictions = forest.predict(val_X)
forest_mae = mean_absolute_error(val_y, forest_predictions)
forest_r2 = r2_score(val_y, forest_predictions)

print(f"Random forest MAE: ${forest_mae:,.0f}")
print(f"Random forest R²: {forest_r2:.3f}")
print(f"MAE reduction vs decision tree: {(mae - forest_mae) / mae:.1%}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

numeric_features = [
    "GrLivArea", "YearBuilt", "BedroomAbvGr", "FullBath", "LotArea",
    "OverallQual", "OverallCond", "YearRemodAdd",
    "TotalBsmtSF", "1stFlrSF", "2ndFlrSF",
    "GarageCars", "GarageArea", "TotRmsAbvGrd",
    "Fireplaces", "BsmtFullBath", "WoodDeckSF", "OpenPorchSF"
]

categorical_features = [
    "Neighborhood", "HouseStyle", "BldgType",
    "ExterQual", "KitchenQual", "Foundation",
    "CentralAir", "GarageType", "SaleCondition"
]

X = train[numeric_features + categorical_features]
y = train["SalePrice"]

train_X, val_X, train_y, val_y = train_test_split(
    X, y, random_state=0
)

# Fill missing values and convert categories into numeric columns.
numeric_pipeline = SimpleImputer(strategy="median")

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

improved_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=500,
        random_state=1,
        n_jobs=-1
    ))
])

# Learn preprocessing and model patterns from training data only.
improved_model.fit(train_X, train_y)
predictions = improved_model.predict(val_X)

new_mae = mean_absolute_error(val_y, predictions)
new_r2 = r2_score(val_y, predictions)

print(f"Updated MAE: ${new_mae:,.0f}")
print(f"Updated R²: {new_r2:.3f}")
print(f"MAE reduction vs original tree: {(32240-new_mae)/32240:.1%}")
print(f"MAE reduction vs first forest: {(23360-new_mae)/23360:.1%}")

In [ ]:
from sklearn.model_selection import KFold, cross_validate

folds = KFold(n_splits=5, shuffle=True, random_state=42)

results = cross_validate(
    improved_model,
    X,
    y,
    cv=folds,
    scoring={
        "mae": "neg_mean_absolute_error",
        "r2": "r2"
    },
    n_jobs=1
)

mae_scores = -results["test_mae"]
r2_scores = results["test_r2"]

for i, (mae, r2) in enumerate(zip(mae_scores, r2_scores), 1):
    print(f"Fold {i}: MAE ${mae:,.0f} | R² {r2:.3f}")

print(f"\nAverage MAE: ${mae_scores.mean():,.0f}")
print(f"MAE standard deviation: ${mae_scores.std():,.0f}")
print(f"Average R²: {r2_scores.mean():.3f}")